# Paper Rebalance Order Analysis

This notebook analyzes the dry-run rebalance orders created from the Random Forest portfolio workflow.

The workflow is:

RF predicted volatility -> portfolio optimizer -> target weights -> dry-run rebalance orders

These are simulated orders only. They do not submit trades to a brokerage account.

# Imports

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Paths

In [2]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

In [3]:
PORTFOLIO_PATH = PROJECT_ROOT / "data" / "processed" / "portfolio_optimization"
WEIGHTS_PATH = PORTFOLIO_PATH / "live_weights"
ORDERS_PATH = PORTFOLIO_PATH / "paper_orders"

In [4]:
weights_files = sorted(WEIGHTS_PATH.glob("target_weights_*.csv"))
orders_files = sorted(ORDERS_PATH.glob("rebalance_orders_*.csv"))

if not weights_files:
    raise FileNotFoundError(f"No target weight files found in {WEIGHTS_PATH}")
if not orders_files:
    raise FileNotFoundError(f"No rebalance order files found in {ORDERS_PATH}")

latest_weights_file = weights_files[-1]
latest_orders_file = orders_files[-1]

latest_weights_file, latest_orders_file

(WindowsPath('C:/Users/sshab/Documents/coding_projects/INFO442-Group-Project/data/processed/portfolio_optimization/live_weights/target_weights_2026-08-06.csv'),
 WindowsPath('C:/Users/sshab/Documents/coding_projects/INFO442-Group-Project/data/processed/portfolio_optimization/paper_orders/rebalance_orders_2026-08-06.csv'))

# Load Latest Target Weights and Orders

In [5]:
weights = pd.read_csv(latest_weights_file, parse_dates=["Date"])
orders = pd.read_csv(latest_orders_file, parse_dates=["Date"])

weights = weights.sort_values("target_weight", ascending=False).reset_index(drop=True)
orders = orders.sort_values("target_weight", ascending=False).reset_index(drop=True)

actionable_orders = orders[orders["status"] == "dry_run"].copy()

In [6]:
weights.head()

,Date,ticker,target_weight
0,2026-08-06,AGG,0.250000
1,2026-08-06,TLT,0.250000
2,2026-08-06,GLD,0.084231
3,2026-08-06,UNH,0.076295
4,2026-08-06,NEE,0.051844


In [7]:
orders.head()

,Date,ticker,target_weight,estimated_price,target_dollars,current_quantity,current_dollars,trade_dollars,side,quantity,estimated_order_dollars,status
0,2026-08-06,AGG,0.250000,98.156616,25000.000000,100.0,9815.661621,15184.338379,buy,154,15116.118896,dry_run
1,2026-08-06,TLT,0.250000,85.911827,25000.000000,50.0,4295.591354,20704.408646,buy,240,20618.838501,dry_run
2,2026-08-06,GLD,0.084231,398.890015,8423.099417,5.0,1994.450073,6428.649343,buy,16,6382.240234,dry_run
3,2026-08-06,UNH,0.076295,327.725800,7629.501429,0.0,0.000000,7629.501429,buy,23,7537.693390,dry_run
4,2026-08-06,NEE,0.051844,79.403030,5184.374789,0.0,0.000000,5184.374789,buy,65,5161.196976,dry_run


# Data Summary

In [8]:
prediction_date = weights["Date"].max().date()
portfolio_value = orders["target_dollars"].sum()
estimated_actionable_order_dollars = actionable_orders["estimated_order_dollars"].abs().sum()
assets_with_positive_weight = int((weights["target_weight"] > 0).sum())
all_current_positions_zero = np.isclose(orders["current_quantity"].fillna(0).sum(), 0)

cash_only_rounding_gap = np.nan
if all_current_positions_zero and (orders["side"] != "sell").all():
    cash_only_rounding_gap = portfolio_value - orders.loc[orders["side"] == "buy", "estimated_order_dollars"].sum()

summary = pd.DataFrame({
    "Metric": [
        "Prediction target date",
        "Portfolio value",
        "Assets considered",
        "Assets with positive target weight",
        "Actionable dry-run orders",
        "Buy orders",
        "Sell orders",
        "Hold rows",
        "Estimated actionable order dollars",
        "Whole-share rounding cash gap",
        "Current positions assumption",
    ],
    "Value": [
        prediction_date,
        f"${portfolio_value:,.2f}",
        len(weights),
        assets_with_positive_weight,
        int((orders["status"] == "dry_run").sum()),
        int(((orders["status"] == "dry_run") & (orders["side"] == "buy")).sum()),
        int(((orders["status"] == "dry_run") & (orders["side"] == "sell")).sum()),
        int((orders["side"] == "hold").sum()),
        f"${estimated_actionable_order_dollars:,.2f}",
        "N/A" if pd.isna(cash_only_rounding_gap) else f"${cash_only_rounding_gap:,.2f}",
        "Cash-only / zero current holdings" if all_current_positions_zero else "Loaded current positions file",
    ],
})

In [9]:
summary

,Metric,Value
0,Prediction target date,2026-08-06
1,Portfolio value,"$100,000.00"
2,Assets considered,20
3,Assets with positive target weight,15
4,Actionable dry-run orders,17
5,Buy orders,15
6,Sell orders,2
7,Hold rows,3
8,Estimated actionable order dollars,"$83,276.21"
9,Whole-share rounding cash gap,N/A


# Target Portfolio Weights

In [10]:
fig = px.bar(
    weights,
    x="target_weight",
    y="ticker",
    orientation="h",
    title=f"Optimized Target Weights for {prediction_date}",
    labels={"target_weight": "Target Weight", "ticker": "Ticker"},
)
fig.update_layout(
    xaxis_tickformat=".0%",
    yaxis={"categoryorder": "total ascending"},
    height=650,
)
fig.show()

In [11]:
positive_weights = weights[weights["target_weight"] > 0].copy()

fig = px.treemap(
    positive_weights,
    path=["ticker"],
    values="target_weight",
    color="target_weight",
    color_continuous_scale="Blues",
    title="Target Portfolio Allocation Treemap",
)
fig.update_traces(texttemplate="%{label}<br>%{value:.1%}")
fig.show()

# Target Dollars

In [12]:
fig = px.bar(
    orders.sort_values("target_dollars", ascending=False),
    x="ticker",
    y="target_dollars",
    color="target_weight",
    title="Target Dollar Allocation by Asset",
    labels={
        "ticker": "Ticker",
        "target_dollars": "Target Dollars",
        "target_weight": "Target Weight",
    },
)
fig.update_layout(yaxis_tickprefix="$", yaxis_tickformat=",.0f")
fig.show()

# Dry-Run Rebalance Orders

In [13]:
if actionable_orders.empty:
    fig = go.Figure()
    fig.add_annotation(
        text="No actionable dry-run orders were generated.",
        xref="paper",
        yref="paper",
        x=0.5,
        y=0.5,
        showarrow=False,
        font=dict(size=16),
    )
    fig.update_layout(title="Estimated Dollar Size of Actionable Dry-Run Orders")
else:
    fig = px.bar(
        actionable_orders.sort_values("estimated_order_dollars", ascending=False),
        x="ticker",
        y="estimated_order_dollars",
        color="side",
        title="Estimated Dollar Size of Actionable Dry-Run Orders",
        labels={
            "ticker": "Ticker",
            "estimated_order_dollars": "Estimated Order Dollars",
            "side": "Order Side",
        },
    )
    fig.update_layout(yaxis_tickprefix="$", yaxis_tickformat=",.0f")

fig.show()

In [14]:
if actionable_orders.empty:
    fig = go.Figure()
    fig.add_annotation(
        text="No actionable dry-run orders were generated.",
        xref="paper",
        yref="paper",
        x=0.5,
        y=0.5,
        showarrow=False,
        font=dict(size=16),
    )
    fig.update_layout(title="Order Quantity vs Estimated Price")
else:
    fig = px.scatter(
        actionable_orders,
        x="estimated_price",
        y="quantity",
        size="estimated_order_dollars",
        color="ticker",
        hover_data=["target_weight", "target_dollars", "side"],
        title="Order Quantity vs Estimated Price",
        labels={
            "estimated_price": "Estimated Price",
            "quantity": "Quantity",
            "estimated_order_dollars": "Estimated Order Dollars",
        },
    )
    fig.update_layout(xaxis_tickprefix="$", xaxis_tickformat=",.2f")

fig.show()

# Buy, Sell, and Hold Breakdown

In [15]:
side_counts = orders["side"].value_counts().reset_index()
side_counts.columns = ["side", "count"]

fig = px.pie(
    side_counts,
    names="side",
    values="count",
    hole=0.45,
    title="Order Action Breakdown",
)
fig.show()

In [16]:
status_counts = orders["status"].value_counts().reset_index()
status_counts.columns = ["status", "count"]

fig = px.bar(
    status_counts,
    x="status",
    y="count",
    title="Order Status Counts",
    labels={"status": "Status", "count": "Rows"},
)
fig.show()

# Current vs Target Dollars

In [17]:
comparison = orders.copy()
comparison_long = comparison.melt(
    id_vars=["ticker"],
    value_vars=["current_dollars", "target_dollars"],
    var_name="portfolio_state",
    value_name="dollars",
)

fig = px.bar(
    comparison_long,
    x="ticker",
    y="dollars",
    color="portfolio_state",
    barmode="group",
    title="Current Dollars vs Target Dollars",
    labels={
        "ticker": "Ticker",
        "dollars": "Dollars",
        "portfolio_state": "Portfolio State",
    },
)
fig.update_layout(yaxis_tickprefix="$", yaxis_tickformat=",.0f")
fig.show()

# Portfolio Concentration

In [18]:
weights_sorted = weights.sort_values("target_weight", ascending=False).copy()
weights_sorted["cumulative_weight"] = weights_sorted["target_weight"].cumsum()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=weights_sorted["ticker"],
    y=weights_sorted["target_weight"],
    name="Target Weight",
))

fig.add_trace(go.Scatter(
    x=weights_sorted["ticker"],
    y=weights_sorted["cumulative_weight"],
    name="Cumulative Weight",
    mode="lines+markers",
    yaxis="y2",
))

fig.update_layout(
    title="Portfolio Concentration and Cumulative Weight",
    yaxis=dict(title="Target Weight", tickformat=".0%"),
    yaxis2=dict(title="Cumulative Weight", tickformat=".0%", overlaying="y", side="right"),
    xaxis_title="Ticker",
    height=550,
)

fig.show()

In [19]:
hhi = (weights["target_weight"] ** 2).sum()
effective_assets = 1 / hhi if hhi > 0 else np.nan
top_5_weight = weights.sort_values("target_weight", ascending=False).head(5)["target_weight"].sum()

concentration_summary = pd.DataFrame({
    "Metric": [
        "Top 5 weight",
        "Herfindahl-Hirschman Index",
        "Effective number of assets",
        "Largest single weight",
    ],
    "Value": [
        f"{top_5_weight:.2%}",
        f"{hhi:.4f}",
        f"{effective_assets:.2f}",
        f"{weights['target_weight'].max():.2%}",
    ],
})

concentration_summary

,Metric,Value
0,Top 5 weight,71.24%
1,Herfindahl-Hirschman Index,0.1502
2,Effective number of assets,6.66
3,Largest single weight,25.00%


# Order Table

In [20]:
display_columns = [
    "Date",
    "ticker",
    "target_weight",
    "estimated_price",
    "target_dollars",
    "current_quantity",
    "current_dollars",
    "trade_dollars",
    "side",
    "quantity",
    "estimated_order_dollars",
    "status",
]

orders_display = orders[display_columns].copy()
orders_display["Date"] = orders_display["Date"].dt.date.astype(str)

orders_display.style.format({
    "target_weight": "{:.2%}",
    "estimated_price": "${:,.2f}",
    "target_dollars": "${:,.2f}",
    "current_dollars": "${:,.2f}",
    "trade_dollars": "${:,.2f}",
    "estimated_order_dollars": "${:,.2f}",
    "quantity": "{:,.4g}",
})

,Date,ticker,target_weight,estimated_price,target_dollars,current_quantity,current_dollars,trade_dollars,side,quantity,estimated_order_dollars,status
0,2026-08-06,AGG,25.00%,$98.16,"$25,000.00",100.000000,"$9,815.66","$15,184.34",buy,154,"$15,116.12",dry_run
1,2026-08-06,TLT,25.00%,$85.91,"$25,000.00",50.000000,"$4,295.59","$20,704.41",buy,240,"$20,618.84",dry_run
2,2026-08-06,GLD,8.42%,$398.89,"$8,423.10",5.000000,"$1,994.45","$6,428.65",buy,16,"$6,382.24",dry_run
3,2026-08-06,UNH,7.63%,$327.73,"$7,629.50",0.000000,$0.00,"$7,629.50",buy,23,"$7,537.69",dry_run
4,2026-08-06,NEE,5.18%,$79.40,"$5,184.37",0.000000,$0.00,"$5,184.37",buy,65,"$5,161.20",dry_run
5,2026-08-06,MSFT,4.65%,$485.32,"$4,645.90",8.000000,"$3,882.55",$763.35,buy,1,$485.32,dry_run
6,2026-08-06,LLY,4.48%,"$1,076.10","$4,482.33",0.000000,$0.00,"$4,482.33",buy,4,"$4,304.39",dry_run
7,2026-08-06,AMZN,3.49%,$232.53,"$3,486.16",4.000000,$930.12,"$2,556.04",buy,10,"$2,325.30",dry_run
8,2026-08-06,XOM,3.44%,$119.38,"$3,436.97",0.000000,$0.00,"$3,436.97",buy,28,"$3,342.60",dry_run
9,2026-08-06,VNQ,3.04%,$87.51,"$3,040.78",0.000000,$0.00,"$3,040.78",buy,34,"$2,975.17",dry_run


# Conclusion

This notebook shows that the project now connects the Random Forest volatility model to a simulated portfolio management workflow. The latest run uses the model's predicted 20-day volatility to build optimized target weights, then converts those target weights into dry-run rebalance orders that can be reviewed before any broker integration.

For the 2026-08-06 target date, the workflow produced a $100,000 simulated target portfolio across 20 considered assets. The minimum-volatility optimizer assigned positive weights to 15 assets, with the largest allocations going to AGG and TLT at the 25% per-asset cap. This is consistent with the risk-focused objective because those bond ETF positions had lower predicted volatility than most equity assets.

After adding the sample current-positions file, the order-generation step now demonstrates true rebalancing behavior instead of only building from cash. It produced 15 dry-run buy orders, 2 dry-run sell orders, and 3 hold rows. The sell orders were for AAPL and V, which had current simulated holdings but received 0% target weight from the optimizer in this run.

The main result is not that these trades should be placed. The important result is that the project can move from real model predictions to auditable portfolio weights and order-level instructions while staying safely in simulation mode. A strong next step would be testing multiple risk profiles or adding an Alpaca paper-account connector that only submits orders after a separate review step.